# NetCDF to DGGS (H3) Conversion

This notebook provides an approach for converting NetCDF climate data to multi-resolution DGGS H3 format.

## Overview

This notebook processes NetCDF climate data to multi-resolution H3 DGGS format for use with [`pydggsapi`](https://github.com/LandscapeGeoinformatics/pydggsapi):

1. Determines optimal H3 resolution from NetCDF grid spacing
2. Processes NetCDF climate data to H3 DGGS format using `xarray` operations
3. Aggregates spatial data from lat/lon grid to H3 cells via `groupby()` mean
4. Generates parent resolutions through cascading H3 aggregation (base → 0)
5. Combines monthly files into continuous timeseries via time concatenation
6. Creates hierarchical Zarr with resolution groups (`res0`, `res1`, ..., `res6`)
7. Applies chunking for efficient data access (e.g.: chunks considering temporal monthly/yearly splits)
8. Extracts variable metadata from NetCDF attributes and augment them with [ClimateData.ca](https://climatedata.ca/variables/) definitions
9. Generates complete `pydggsapi` configuration with spatiotemporal collection metadata

### Output Structure

After processing, your output directory will contain:
```
outputs/
├── dggs/
│   └── h3/
│       ├── res=0/                               # Coarsest resolution
│       │   └── {file}_h3_res=0.zarr
│       ├── res=1/
│       │   └── {file}_h3_res=1.zarr
│       ├── ...
│       ├── res=6/                               # Base resolution (finest)
│       │   └── {file}_h3_res=6.zarr
│       └── canada_climate_h3_dggs.zarr/         # Final combined hierarchical Zarr
│           ├── .zmetadata                       # Consolidated metadata
│           ├── res0/                            # Group for resolution 0
│           │   ├── ssp126_prcptot_p10/          # Variable arrays
│           │   │   ├── .zarray                  # Array metadata (chunks, dtype, etc.)
│           │   │   └── 0.0, 0.1, ...            # Chunk files
│           │   └── ...
│           ├── res1/                            # Group for resolution 1
│           ├── ...
│           └── res6/                            # Group for resolution 6 (finest)
│               ├── ssp126_prcptot_p10/
│               │   ├── .zarray                  # chunks: (207466, 12) for monthly data
│               │   └── 0.0, 0.1, ..., 0.150     # 151 chunks (1 per year)
│               └── ...
├── pydggsapi_config.json                        # pydggsapi configuration
└── parent_resolutions_summary.json              # Parent aggregation stats
```

Hierarchical Zarr structure details:
- Each resolution is stored as a separate group (`res0`, `res1`, ..., `res6`)
- Monthly data (`prcptot`, `tx_max`): Concatenated along time dimension
  - Shape: (N_cells, 1812) where 1812 = 151 years × 12 months
  - Chunks: (N_cells, 12) = 151 yearly chunks
  - Time coords: [1950-01-15, 1950-02-15, ..., 2100-12-15]
- Yearly data (`frost_days`, `ice_days`): Single time dimension
  - Shape: (N_cells, 151) where 151 = years 1950-2100
  - Chunks: (N_cells, 151) = 1 chunk
  - Time coords: [1950, 1951, ..., 2100]

### Cache Management

To reprocess data:
- Clear H3 grid cache: Remove files in `{CACHE_DIR}/h3_grid_*.npy`
- Clear metadata cache: Remove `{CACHE_DIR}/climate_variable_metadata.json`
- Clear processed Zarr files: Remove directories in `{OUTPUT_DIR}/dggs/h3/res=*/`
- Clear final combined Zarr store: Remove `{FINAL_ZARR_PATH}`

---

## Technical Approach

**⚠️ Important Consideration ⚠️**

Both NetCDF and Zarr are chunked array formats. We leverage `xarray` to skip intermediate conversions with other formats.
However, we assume that the NetCDF files we are interested in are mostly aligned spatiotemporally already to skip certain steps.
For example, chucking will take into consideration that the time dimension is consistent across all files,
so the same chunking strategy applies identically for all DGGS resolutions to avoid fragmentation.

**Strategy**:
1. Compute `(lat, lon) → h3_id` mapping once per grid
2. Add H3 as coordinate to `xarray.Dataset`
3. Use `xarray`'s `groupby()` to aggregate (time, lat, lon) → (time, h3_id)
4. Write directly to Zarr with optimal chunking

### Important Assumptions & Design Decisions

**Spatiotemporal Alignment**:
- NetCDF files are assumed to be mostly aligned spatiotemporally
- This allows skipping redundant alignment/resampling steps
- All files within an index/aggregation type share the same temporal grid

**Chunking Strategy**:
- **Spatial dimension (`h3_id`)**: Single chunk containing all H3 cells
  - Minimizes chunk overhead (metadata, compression headers)
  - Optimal for spatial queries (common in DGGS use cases)
  - Avoids fragmentation from `groupby()` operations

- **Temporal dimension (`time`)**: Single chunk containing all timesteps
  - Maintains consistency across all DGGS resolutions
  - Parent DGGS resolutions inherit temporal chunking from base resolution
  - Ensures uniform query performance across resolution hierarchy

**Why Chunking Matters**:
- Without explicit chunking, `groupby()` creates fragmented chunks (1000s of small chunks)
- Fragmentation causes massive storage bloat (~100x more storage for the same data) and poor read performance
- Small chunks have poor compression ratios and high metadata overhead
- Consistent chunking across resolutions enables predictable and efficient query performance


## Package Setup

### Installation
Only run this cell if packages are missing. Most should be available in the ogc-dggs environment.

In [1]:
%%capture
!pip install xarray zarr h3 numpy pandas tqdm requests netcdf4

### Imports

In [2]:
import datetime
from pathlib import Path
from tqdm.auto import tqdm
import h3
import json
import numpy as np
import os
import pandas as pd
import requests
import time
import warnings
import xarray as xr
import zarr

warnings.filterwarnings("ignore", message="Consolidated metadata is currently not part in the Zarr format 3 specification")

## Configuration

Define all paths and processing parameters.


In [3]:
def ensure_directory(path):
    """
    Ensure directory exists, creating it or following symlinks as needed.
    """
    path = Path(path)

    if path.is_symlink():
        target = path.resolve()
        target.mkdir(parents=True, exist_ok=True)
        return target
    else:
        path.mkdir(parents=True, exist_ok=True)
        return path


In [14]:
# Climate variables to process
CLIMATE_VARIABLES = ["prcptot", "tx_max"]  # ["prcptot", "tx_max", "ice_days", "frost_days"]

# Aggregation types: Monthly (MS) or Yearly (YS)
AGG_TYPE = "MS"

# Directory paths
BASE_DIR = Path(os.getenv("CLIMATE_DATA_BASE_DIR", "data/allyears"))
CACHE_DIR = ensure_directory(os.getenv("CLIMATE_CACHE_DIR", "cache"))
OUTPUT_DIR = ensure_directory(os.getenv("CLIMATE_OUTPUT_DIR", "outputs"))

# DGGS output hierarchy (consistent structure for all zarr outputs)
DGGS_OUTPUT_DIR = OUTPUT_DIR / "dggs"
H3_OUTPUT_DIR = DGGS_OUTPUT_DIR / "h3"

# Final output paths
FINAL_ZARR_PATH = H3_OUTPUT_DIR / "canada_climate_h3_dggs.zarr"
FINAL_CONFIG_PATH = OUTPUT_DIR / "pydggsapi_config.json"

print(f"Configuration loaded:")
print(f"  Climate variables: {CLIMATE_VARIABLES}")
print(f"  Aggregation type: {AGG_TYPE}")
print(f"  Base directory: {BASE_DIR}")
print(f"  Cache directory: {CACHE_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  DGGS output hierarchy: {H3_OUTPUT_DIR}")
print(f"  Final zarr output: {FINAL_ZARR_PATH}")
print(f"  Final config output: {FINAL_CONFIG_PATH}")


Configuration loaded:
  Climate variables: ['prcptot', 'tx_max']
  Aggregation type: MS
  Base directory: data/allyears
  Cache directory: /misc/scratch13/OGC11/canada-climate/cache
  Output directory: /misc/scratch13/OGC11/canada-climate/outputs
  DGGS output hierarchy: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3
  Final zarr output: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3/canada_climate_h3_dggs.zarr
  Final config output: /misc/scratch13/OGC11/canada-climate/outputs/pydggsapi_config.json


## Step 1: Determine Optimal H3 Resolution

Analyze NetCDF grid spacing and find the best matching H3 resolution.


In [5]:
def analyze_netcdf_resolution(nc_files):
    """
    Analyze spatial resolution of NetCDF files and determine optimal H3 resolution.

    Returns:
        tuple: (h3_resolution, grid_info_dict)
    """
    print(f"Analyzing {len(nc_files)} NetCDF files for spatial resolution...")

    resolutions = []
    for f in tqdm(nc_files, desc="Analyzing files"):
        ds = xr.open_dataset(f, decode_timedelta=False)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]

        # Estimate resolution (in degrees)
        lat_res = float(np.abs(lat[1] - lat[0]))
        lon_res = float(np.abs(lon[1] - lon[0]))

        # Approximate km using haversine formula
        mean_lat = float(lat.mean())
        earth_radius_km = 6371.0
        lat_km = lat_res * (np.pi/180) * earth_radius_km
        lon_km = lon_res * (np.pi/180) * earth_radius_km * np.cos(mean_lat * np.pi/180)

        resolutions.append((f, lat_km, lon_km))
        ds.close()

    # Find best (finest) resolution
    best_file, best_lat_km, best_lon_km = min(resolutions, key=lambda x: min(x[1], x[2]))
    print(f"\nBest spatial resolution: {best_lat_km:.2f} km x {best_lon_km:.2f} km")
    print(f"  from: {Path(best_file).name}")

    # Map NetCDF Grid to H3 DGGS
    # Find the finest H3 resolution where edge <= grid resolution
    h3_resolution = None
    for res in range(16):
        h3_edge_km = h3.average_hexagon_edge_length(res, "km")
        if h3_edge_km <= min(best_lat_km, best_lon_km):
            h3_resolution = res
            break

    if h3_resolution is None:
        h3_resolution = 15
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")
    else:
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")

    print(f"\nSelected H3 resolution: {h3_resolution} (edge ~{h3_edge_km:.3f} km)")
    print(f"  Reasoning: H3 res {h3_resolution} has edge {h3_edge_km:.3f} km <= grid {min(best_lat_km, best_lon_km):.2f} km")

    return h3_resolution, {
        'best_lat_km': best_lat_km,
        'best_lon_km': best_lon_km,
        'h3_edge_km': h3_edge_km,
        'best_file': best_file
    }


## Step 2: Helper Functions

In [6]:
def get_dggs_output_path(h3_resolution, filename=None):
    """
    Generate consistent output path for DGGS H3 zarr files.
    """
    output_dir = H3_OUTPUT_DIR / f"res={h3_resolution}"

    if filename:
        return output_dir / filename
    return output_dir


Compute H3 variables for entire lat/lon grid at once

In [40]:
def compute_h3_grid(lat, lon, resolution):
    """
    Compute H3 cell IDs for a lat/lon grid (vectorized).
    Returns 2D array of H3 cell IDs as uint64.
    """
    lat_grid, lon_grid = np.meshgrid(lat, lon, indexing='ij')
    lat_flat = lat_grid.ravel()
    lon_flat = lon_grid.ravel()

    # Still need list comprehension for h3 library, but vectorized the grid creation
    h3_cells = np.array([
        int(h3.latlng_to_cell(float(la), float(lo), resolution), 16)
        for la, lo in zip(lat_flat, lon_flat)
    ], dtype=np.uint64)

    return h3_cells.reshape(lat_grid.shape)


In [41]:
def build_zarr_encoding(dataset):
    """
    Return per-variable chunk metadata for writing to Zarr.

    Extracts uniform chunk sizes from dask arrays. Returns None for encoding
    if chunks are not uniform, letting xarray handle it automatically.
    """
    encoding = {}
    for var in dataset.data_vars:
        if hasattr(dataset[var].data, 'chunks') and dataset[var].data.chunks:
            # Check if all chunks in each dimension are uniform
            uniform_chunks = []
            for dim_chunks in dataset[var].data.chunks:
                if isinstance(dim_chunks, tuple):
                    # Check if all chunks are the same size
                    unique_sizes = set(dim_chunks)
                    if len(unique_sizes) == 1:
                        # Uniform chunks
                        uniform_chunks.append(dim_chunks[0])
                    else:
                        # Non-uniform chunks - don't specify encoding
                        uniform_chunks = None
                        break
                else:
                    uniform_chunks.append(dim_chunks)

            if uniform_chunks:
                encoding[var] = {"chunks": tuple(uniform_chunks)}
        # If no chunks or non-uniform, don't specify encoding for this variable
    return encoding


## Step 3: Process NetCDF Files to H3 Zarr Format

Convert NetCDF climate data to H3 DGGS format. Tests with a single file first to determine
optimal H3 resolution, then processes all configured climate variables.


In [42]:
def process_netcdf_to_h3_single_resolution(nc_file, h3_resolution, skip_existing=True):
    """
    Process NetCDF to H3 DGGS at single resolution using xarray operations.
    """
    timings = {}
    start_total = time.time()

    print(f"\nProcessing: {Path(nc_file).name}")

    # Create resolution-specific output directory using helper function
    output_dir = get_dggs_output_path(h3_resolution)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_zarr = output_dir / f"{Path(nc_file).stem}_h3_res={h3_resolution}.zarr"

    # Check if already exists
    if skip_existing and output_zarr.exists():
        print(f"  ⏭️ Output already exists, skipping")
        ds_existing = xr.open_zarr(output_zarr, decode_timedelta=False)
        timings['total'] = 0
        timings['skipped'] = True
        return ds_existing, timings, output_zarr

    # Open NetCDF file (no chunking yet, keep it simple)
    t0 = time.time()
    ds = xr.open_dataset(nc_file, decode_timedelta=False)
    timings['open_netcdf'] = time.time() - t0

    # Get latitude and longitude variables
    lat = ds['lat'].values if 'lat' in ds else ds['latitude'].values
    lon = ds['lon'].values if 'lon' in ds else ds['longitude'].values

    # Compute or load cached H3 grid mapping
    grid_hash = hash((tuple(lat), tuple(lon), h3_resolution))
    cache_file = Path(CACHE_DIR) / f"h3_grid_{grid_hash}.npy"

    t0 = time.time()
    if cache_file and cache_file.exists():
        print(f"  Loading cached H3 grid...")
        h3_grid = np.load(cache_file)
    else:
        print(f"  Computing H3 grid (res={h3_resolution})...")
        h3_grid = compute_h3_grid(lat, lon, h3_resolution)
        if cache_file:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_file, h3_grid)
    timings['h3_grid'] = time.time() - t0

    # Add H3 index as a coordinate
    print(f"  Adding H3 coordinate to dataset...")
    t0 = time.time()
    ds = ds.assign_coords({'h3_index': (('lat', 'lon'), h3_grid)})
    timings['assign_coords'] = time.time() - t0

    # Stack spatial dimensions: (time, lat, lon) → (time, spatial)
    print(f"  Stacking spatial dimensions...")
    t0 = time.time()
    ds_stacked = ds.stack(spatial=['lat', 'lon'])
    timings['stack'] = time.time() - t0

    # Drop NaN values (ocean/missing data)
    print(f"  Dropping NaN values...")
    t0 = time.time()
    ds_stacked = ds_stacked.dropna(dim='spatial', how='all')
    timings['dropna'] = time.time() - t0

    # Group by H3 and aggregate: (time, spatial) → (time, h3_id)
    print(f"  Aggregating by H3 cell...")
    t0 = time.time()
    ds_h3 = ds_stacked.groupby('h3_index').mean()
    timings['groupby_aggregate'] = time.time() - t0

    # Rename dimension for clarity
    ds_h3 = ds_h3.rename({'h3_index': 'h3_id'})

    # Rechunk for efficient storage
    # Spatial: all h3 cells in one chunk for minimal overhead
    # Temporal: all timesteps in one chunk to maintain consistency across resolutions
    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
        'time': ds_h3.sizes['time'] if 'time' in ds_h3.sizes else -1
    }
    ds_h3 = ds_h3.chunk(optimal_chunks)

    encoding = build_zarr_encoding(ds_h3)

    print(f"  Writing to Zarr: {output_zarr}")
    t0 = time.time()
    ds_h3.to_zarr(output_zarr, mode='w', encoding=encoding, compute=True)
    timings['write_zarr'] = time.time() - t0

    ds.close()

    timings['total'] = time.time() - start_total

    print(f"  ⏱️ Total time: {timings['total']:.2f}s")
    print(f"     ├─ H3 grid: {timings['h3_grid']:.2f}s")
    print(f"     ├─ Stack/aggregate: {timings['stack'] + timings['groupby_aggregate']:.2f}s")
    print(f"     └─ Write Zarr: {timings['write_zarr']:.2f}s")

    return ds_h3, timings, output_zarr


In [43]:
# Find test files
var = CLIMATE_VARIABLES[0]
test_files = list(BASE_DIR.glob(f"**/*{var}*.nc"))
if test_files:
    print(f"Found {len(test_files)} files")

    # Determine optimal H3 resolution from the data
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(test_files[:5])  # Analyze first 5 files

    print(f"\n{'='*60}")
    print(f"Will use H3 resolution: {H3_RESOLUTION}")
    print(f"{'='*60}\n")

    # Process first file as test
    test_file = test_files[0]
    print(f"Test file: {test_file}")

    # Process it (will skip if already exists)
    result, timings, output_path = process_netcdf_to_h3_single_resolution(
        test_file,
        H3_RESOLUTION,
        skip_existing=True
    )

    print(f"\n✅ Test completed successfully!")
    print(f"Result shape: {result.dims}")
    print(f"Variables: {list(result.data_vars)}")
    print(f"Output: {output_path}")
else:
    print("No test files found")


Found 17 prcptot files
Analyzing 5 NetCDF files for spatial resolution...


Analyzing files:   0%|          | 0/5 [00:00<?, ?it/s]


Best spatial resolution: 9.27 km x 4.31 km
  from: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Selected H3 resolution: 6 (edge ~3.725 km)
  Reasoning: H3 res 6 has edge 3.725 km <= grid 4.31 km

Will use H3 resolution: 6

Test file: data/allyears/prcptot/MS/prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
  ⏭️ Output already exists, skipping

✅ Test completed successfully!
Result shape: FrozenMappingWarningOnValuesAccess({'h3_id': 207466, 'time': 151})
Variables: ['ssp126_prcptot_p10', 'ssp126_prcptot_p50', 'ssp126_prcptot_p90', 'ssp245_prcptot_p10', 'ssp245_prcptot_p50', 'ssp245_prcptot_p90', 'ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp585_prcptot_p10', 'ssp585_prcptot_p50', 'ssp585_prcptot_p90']
Output: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3/res=6/prcptot_

## Step 4: Process All Climate Variables

Process all monthly NetCDF files for all configured climate variables at the determined H3 resolution.


In [44]:
def process_all_nc_files(nc_files, h3_resolution):
    """
    Process all NetCDF files.

    Args:
        nc_files: List of NetCDF file paths to process
        h3_resolution: H3 resolution to use

    Returns:
        dict: Summary statistics and timing info
    """
    all_timings = []
    processed_files = []

    all_nc_files = sorted(set(nc_files))  # Remove duplicates

    print(f"\n{'='*70}")
    print(f"Processing {len(all_nc_files)} NetCDF files at H3 resolution {h3_resolution}")
    print(f"{'='*70}\n")

    start_total = time.time()

    for i, nc_file in enumerate(all_nc_files, 1):
        print(f"\n[{i}/{len(all_nc_files)}] {nc_file.name}")

        try:
            result, timings, output_path = process_netcdf_to_h3_single_resolution(
                nc_file,
                h3_resolution,
                skip_existing=True
            )

            # Only add timing if actually processed (not skipped)
            if not timings.get('skipped', False):
                all_timings.append({
                    'file': nc_file.name,
                    'variables': list(result.data_vars),
                    'n_variables': len(result.data_vars),
                    'n_cells': result.sizes.get('h3_id', 0),
                    'n_timesteps': result.sizes.get('time', 0),
                    **timings
                })

            processed_files.append(output_path)

        except Exception as e:
            print(f"  ❌ Error processing {nc_file.name}: {e}")
            continue

    total_time = time.time() - start_total

    print(f"\n{'='*70}")
    print(f"✅ Processing complete!")
    print(f"{'='*70}")
    print(f"Total time: {total_time/60:.2f} minutes ({total_time:.1f}s)")
    print(f"Processed: {len(processed_files)} files")
    print(f"Average time per file: {total_time/max(len(all_timings), 1):.2f}s")

    # Save timing summary
    timing_summary = {
        'h3_resolution': h3_resolution,
        'total_files': len(all_nc_files),
        'processed_files': len(processed_files),
        'total_time_seconds': total_time,
        'timings': all_timings,
        'timestamp': datetime.datetime.now().isoformat()
    }

    summary_file = Path(OUTPUT_DIR) / f"timing_summary_res={h3_resolution}.json"
    with open(summary_file, 'w') as f:
        json.dump(timing_summary, f, indent=2)

    print(f"Timing summary saved to: {summary_file}")

    return timing_summary


In [45]:
# Find all files matching variables and aggregation types
all_files = []
for var in CLIMATE_VARIABLES:
    pattern = BASE_DIR / var / AGG_TYPE / "*.nc"
    files = list(pattern.parent.glob("*.nc")) if pattern.parent.exists() else []
    all_files.extend(files)
all_files = sorted(set(all_files))

if all_files:
    # Determine H3 resolution by sampling first 10 files
    sample_size = min(10, len(all_files))
    print(f"\nDetermining optimal H3 resolution from {sample_size} sample file(s)...\n")
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(all_files[:sample_size])

    print(f"\n{'='*70}")
    print(f"CONFIGURATION")
    print(f"{'='*70}")
    print(f"Climate variables: {CLIMATE_VARIABLES}")
    print(f"Aggregation type: {AGG_TYPE}")
    print(f"H3 resolution: {H3_RESOLUTION}")
    print(f"Total files to process: {len(all_files)}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

    # Process all files
    summary = process_all_nc_files(
        nc_files=all_files,
        h3_resolution=H3_RESOLUTION
    )
else:
    print("No NetCDF files found!")



Determining optimal H3 resolution from 10 sample file(s)...

Analyzing 10 NetCDF files for spatial resolution...


Analyzing files:   0%|          | 0/10 [00:00<?, ?it/s]


Best spatial resolution: 9.27 km x 4.31 km
  from: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Selected H3 resolution: 6 (edge ~3.725 km)
  Reasoning: H3 res 6 has edge 3.725 km <= grid 4.31 km

CONFIGURATION
Climate variables: ['prcptot', 'tx_max']
Aggregation type: MS
H3 resolution: 6
Total files to process: 24
Output directory: /misc/scratch13/OGC11/canada-climate/outputs


Processing 24 NetCDF files at H3 resolution 6


[1/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
  ⏭️ Output already exists, skipping

[2/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc
  ⏭️ Output already exists, skipping

[3/24] prcptot_MS_MBCn+PCIC-Blend_historical+a

## Step 5: Aggregate to Parent DGGS Resolutions

Generate coarser resolution levels by aggregating from base resolution down to resolution 0.
Each parent resolution aggregates from its immediate child resolution using H3's built-in
parent-child hierarchy, cascading from finest to coarsest.


In [46]:
def aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr):
    """
    Aggregate H3 data from higher resolution to a parent resolution.

    Args:
        source_zarr: Path to source Zarr store (higher resolution)
        parent_res: Target parent resolution (lower number = coarser)
        output_zarr: Output Zarr path

    Returns:
        xarray Dataset at parent resolution
    """
    print(f"  Aggregating to parent resolution {parent_res}...")

    # Open source data
    ds = xr.open_zarr(source_zarr, decode_timedelta=False)

    # Get current H3 IDs (uint64)
    h3_ids = ds['h3_id'].values

    # Convert to parent H3 IDs
    print(f"    Converting {len(h3_ids)} cells to parent res...")
    parent_h3_ids = np.array([
        int(h3.cell_to_parent(h3.int_to_str(h3_id), parent_res), 16)
        for h3_id in h3_ids
    ], dtype=np.uint64)

    # Add parent H3 ID as coordinate
    ds = ds.assign_coords({'h3_parent': ('h3_id', parent_h3_ids)})

    # Group by parent and aggregate
    print(f"    Grouping and aggregating...")
    ds_parent = ds.groupby('h3_parent').mean()
    ds_parent = ds_parent.rename({'h3_parent': 'h3_id'})

    # Rechunk for efficient storage, preserving temporal chunking from source
    # This ensures consistency across all resolutions
    source_time_chunks = ds.chunks.get('time', None) if hasattr(ds, 'chunks') else None

    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
    }

    # Preserve time chunking from source if available, otherwise use all timesteps
    if 'time' in ds_parent.sizes:
        if source_time_chunks and len(source_time_chunks) > 0:
            # Use same time chunking as source
            optimal_chunks['time'] = source_time_chunks[0]
        else:
            # Fallback: all timesteps in one chunk
            optimal_chunks['time'] = ds_parent.sizes['time']

    ds_parent = ds_parent.chunk(optimal_chunks)

    encoding = build_zarr_encoding(ds_parent)

    print(f"    Writing to {output_zarr}")
    ds_parent.to_zarr(output_zarr, mode='w', encoding=encoding, compute=True)

    ds.close()

    return ds_parent

def process_all_parent_resolutions(base_resolution):
    """
    Process all parent resolutions from base_resolution down to 0.

    Args:
        base_resolution: Starting (finest) H3 resolution

    Returns:
        dict: Summary of processed resolutions
    """
    # Find source files using helper function
    source_path = get_dggs_output_path(base_resolution)
    source_files = sorted(source_path.glob(f"*_h3_res={base_resolution}.zarr"))

    print(f"\n{'='*70}")
    print(f"Aggregating {len(source_files)} variables to parent resolutions")
    print(f"Base resolution: {base_resolution} → Target: 0-{base_resolution-1}")
    print(f"{'='*70}\n")

    summary = {}

    for parent_res in range(base_resolution - 1, -1, -1):
        print(f"\n{'='*70}")
        print(f"Processing resolution {parent_res}")
        print(f"{'='*70}")

        res_start = time.time()
        # Use helper function for consistent output path
        output_dir = get_dggs_output_path(parent_res)
        output_dir.mkdir(parents=True, exist_ok=True)

        processed_count = 0

        for source_file in tqdm(source_files, desc=f"Res {parent_res}"):
            # Determine source for this resolution
            if parent_res == base_resolution - 1:
                # First parent level: aggregate from base resolution
                source_zarr = source_file
            else:
                # Subsequent levels: aggregate from previous parent using helper function
                prev_dir = get_dggs_output_path(parent_res + 1)
                source_zarr = prev_dir / source_file.name.replace(
                    f"_h3_res={base_resolution}.zarr",
                    f"_h3_res={parent_res + 1}.zarr"
                )

                if not source_zarr.exists():
                    print(f"⚠️ Warning: Source not found: {source_zarr}")
                    continue

            # Output file for this parent resolution
            output_zarr = output_dir / source_file.name.replace(
                f"_h3_res={base_resolution}.zarr",
                f"_h3_res={parent_res}.zarr"
            )

            if output_zarr.exists():
                print(f"⏭️ Skipping {source_file.name} (already processed)")
                continue

            try:
                aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr)
                processed_count += 1
            except Exception as e:
                print(f"❌ Error processing {source_file.name} to res {parent_res}: {e}")
                continue

        res_time = time.time() - res_start
        summary[parent_res] = {
            'processed': processed_count,
            'time_seconds': res_time
        }

        print(f"\n✅ Resolution {parent_res} complete: {processed_count} variables in {res_time:.1f}s")

    return summary


In [47]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"Starting parent resolution aggregation from resolution {H3_RESOLUTION}")

    parent_summary = process_all_parent_resolutions(base_resolution=H3_RESOLUTION)

    print(f"\n{'='*70}")
    print(f"✅ All parent resolutions complete!")
    print(f"{'='*70}")

    for res, info in sorted(parent_summary.items(), reverse=True):
        print(f"Resolution {res}: {info['processed']} variables in {info['time_seconds']:.1f}s")

    # Save summary
    summary_file = OUTPUT_DIR / "parent_resolutions_summary.json"
    with open(summary_file, mode="w", encoding="utf-8") as f:
        json.dump(parent_summary, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
else:
    print("H3_RESOLUTION not defined. Run previous cells first.")


Starting parent resolution aggregation from resolution 6

Aggregating 25 variables to parent resolutions
Base resolution: 6 → Target: 0-5


Processing resolution 5


Res 5:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 4:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

...

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 1:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 0:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

## Step 6: Create Hierarchical Zarr with Resolution Groups

Combine individual Zarr files across all resolutions into a single hierarchical Zarr store.
Each resolution is stored as a separate group (`res0`, `res1`, ..., `res6`) for efficient
multi-resolution queries via `pydggsapi`.


In [56]:
def combine_zarr_files_to_grouped_zarr(output_base_dir, h3_resolution, final_output_path, climate_variables=None):
    """
    Combine individual Zarr files across all resolutions into a single hierarchical Zarr
    store with resolution-based groups for `pydggsapi`.

    Args:
        output_base_dir: Base directory containing dggs/h3/res={N}/ subdirectories
        h3_resolution: Maximum H3 resolution (base resolution)
        final_output_path: Path for final combined Zarr store
        climate_variables: List of climate variables to include (optional, includes all if None)

    Returns:
        dict: Mapping of resolution to group names
    """
    print(f"\n{'='*70}")
    print(f"Combining Zarr files into hierarchical Zarr structure")
    print(f"{'='*70}\n")

    final_output_path = Path(final_output_path)
    base_path = Path(output_base_dir) / "dggs" / "h3"

    # Remove existing output if it exists
    if final_output_path.exists():
        import shutil
        shutil.rmtree(final_output_path)

    group_mapping = {}

    # Process each resolution from 0 to h3_resolution
    for res in range(h3_resolution + 1):
        res_dir = base_path / f"res={res}"

        if not res_dir.exists():
            print(f"⚠️ Resolution {res} directory not found: {res_dir}")
            continue

        all_zarr_files = sorted(res_dir.glob("*.zarr"))

        # Filter by climate variables if specified
        if climate_variables:
            zarr_files = []
            for zarr_file in all_zarr_files:
                # Check if filename contains any of the climate variables
                filename_lower = zarr_file.name.lower()
                if any(var.lower() in filename_lower for var in climate_variables):
                    zarr_files.append(zarr_file)

            if len(zarr_files) < len(all_zarr_files):
                print(f"  Filtered {len(all_zarr_files)} files → {len(zarr_files)} files matching {climate_variables}")
        else:
            zarr_files = all_zarr_files

        if not zarr_files:
            print(f"⚠️ No Zarr files found for resolution {res}")
            continue

        print(f"\nProcessing resolution {res}: {len(zarr_files)} file(s)")

        # Group files by their actual time coordinates
        # Files with identical time coordinates have same temporal frequency and alignment
        # (e.g., all January files, all February files, all yearly files with same years)
        print(f"  Loading and grouping files by time coordinates...")
        files_by_time_coords = {}

        for zarr_file in tqdm(zarr_files, desc=f"  Loading res {res}"):
            try:
                ds = xr.open_zarr(zarr_file, decode_timedelta=False)

                # Create hashable key from time coordinates (size, first, last)
                time_vals = tuple(ds['time'].values)
                time_key = (len(time_vals), time_vals[0], time_vals[-1])

                if time_key not in files_by_time_coords:
                    files_by_time_coords[time_key] = []
                files_by_time_coords[time_key].append((zarr_file, ds))

            except Exception as e:
                print(f"    ⚠️ Failed to open {zarr_file.name}: {e}")
                continue

        if not files_by_time_coords:
            print(f"  ⚠️ No valid datasets for resolution {res}")
            continue

        print(f"  Found {len(files_by_time_coords)} unique time coordinate group(s):")
        for time_key, files in files_by_time_coords.items():
            tsize, first, last = time_key
            print(f"    - {tsize} timesteps ({first} to {last}): {len(files)} file(s)")

        # Step 1: Within each time group, merge variables (same time, different variables)
        merged_by_time = []

        for time_key, file_ds_pairs in sorted(files_by_time_coords.items()):
            datasets = [ds for _, ds in file_ds_pairs]
            tsize, first, last = time_key

            print(f"  Merging {len(datasets)} dataset(s) with same time coords...")

            if len(datasets) == 1:
                merged = datasets[0]
            else:
                # Same time coordinates → merge by variables
                merged = xr.merge(datasets, compat='override')

            print(f"    → {len(merged.data_vars)} variables")
            merged_by_time.append(merged)

        # Step 2: Concatenate different time groups along time dimension
        if len(merged_by_time) == 1:
            combined_ds = merged_by_time[0]
            print(f"  Single time group - no time concatenation needed")
        else:
            print(f"  Concatenating {len(merged_by_time)} time groups along time dimension...")
            combined_ds = xr.concat(merged_by_time, dim='time', combine_attrs='override', coords='all')
            combined_ds = combined_ds.sortby('time')

        # Rechunk to ensure consistent chunking for this resolution
        # Spatial: all h3 cells in one chunk (no spatial locality in H3 integer ordering)
        # Temporal: chunk by year for monthly data (12 months), single chunk for yearly data
        time_size = combined_ds.sizes.get('time', 0)
        if time_size > 1000:
            # Monthly data (151 years × 12 months = 1812 timesteps)
            # Chunk by year (12 months) for efficient seasonal/annual queries
            time_chunk = 12
            print(f"  Detected monthly data: {time_size} timesteps → yearly chunks (12 months)")
        elif time_size > 100:
            # Yearly data (151 timesteps)
            # Keep as single chunk
            time_chunk = -1
            print(f"  Detected yearly data: {time_size} timesteps → single chunk")
        else:
            # Small dataset, single chunk
            time_chunk = -1

        optimal_chunks = {
            'h3_id': -1,  # Single chunk for all h3 cells at this resolution
            'time': time_chunk
        }
        print(f"  Rechunking to: {optimal_chunks}")
        combined_ds = combined_ds.chunk(optimal_chunks)
        encoding = build_zarr_encoding(combined_ds)

        # Save to Zarr group
        group_name = f"res{res}"
        print(f"  Writing to group '{group_name}'...")
        print(f"    Variables: {len(combined_ds.data_vars)}")
        print(f"    H3 cells: {combined_ds.sizes.get('h3_id', 0):,}")
        print(f"    Timesteps: {combined_ds.sizes.get('time', 0)}")

        # Write to Zarr using the group parameter
        # Use compute=True to ensure rechunking is applied and chunks are properly stored
        # Use consolidated=False here because we're writing multiple groups incrementally
        # Consolidation happens once at the end after ALL resolution groups are written
        # Use safe_chunks=False to allow writing when encoding chunks don't perfectly align with dask chunks
        mode = 'w' if res == 0 else 'a'
        combined_ds.to_zarr(
            final_output_path,
            group=group_name,
            mode=mode,
            consolidated=False,
            encoding=encoding,
            compute=True,
            safe_chunks=False
        )

        group_mapping[str(res)] = group_name

        # Close all opened datasets to prevent file handle leaks
        for time_key, file_ds_pairs in files_by_time_coords.items():
            for _, ds in file_ds_pairs:
                try:
                    ds.close()
                except:
                    pass

        try:
            combined_ds.close()
        except:
            pass

    # Consolidate metadata once at the end for all groups
    # This creates a single .zmetadata file with metadata for all resolution groups
    # Making subsequent reads much faster (single metadata read instead of per-group)
    print(f"\n{'='*70}")
    print(f"Consolidating metadata...")
    print(f"{'='*70}\n")
    zarr.consolidate_metadata(final_output_path)

    print(f"✅ Hierarchical Zarr created with {len(group_mapping)} resolution group(s)")
    print(f"   Groups: {list(group_mapping.values())}")

    return group_mapping


In [57]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"\n{'='*70}")
    print(f"CREATING FINAL HIERARCHICAL ZARR")
    print(f"{'='*70}\n")

    group_mapping = combine_zarr_files_to_grouped_zarr(
        output_base_dir=OUTPUT_DIR,
        h3_resolution=H3_RESOLUTION,
        final_output_path=FINAL_ZARR_PATH,
        climate_variables=CLIMATE_VARIABLES
    )

    print(f"\n✅ Hierarchical Zarr created: {FINAL_ZARR_PATH}")
    print(f"   Resolution groups: {list(group_mapping.values())}")

else:
    print("H3_RESOLUTION not defined. Run previous cells first.")



CREATING FINAL HIERARCHICAL ZARR


Combining zarr files into hierarchical zarr structure

  Filtered 25 files → 24 files matching ['prcptot', 'tx_max']

Processing resolution 0: 24 file(s)
  Loading and grouping files by time coordinates...


  Loading res 0:   0%|          | 0/24 [00:00<?, ?it/s]

  Found 12 unique time coordinate group(s):
    - 151 timesteps (1950-01-01T00:00:00.000000000 to 2100-01-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-02-01T00:00:00.000000000 to 2100-02-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-03-01T00:00:00.000000000 to 2100-03-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-04-01T00:00:00.000000000 to 2100-04-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-05-01T00:00:00.000000000 to 2100-05-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-06-01T00:00:00.000000000 to 2100-06-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-07-01T00:00:00.000000000 to 2100-07-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-08-01T00:00:00.000000000 to 2100-08-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-09-01T00:00:00.000000000 to 2100-09-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-10-01T00:00:00.000000000 to 2100-10-01T00:00:00.000000000): 2 file

  Loading res 1:   0%|          | 0/24 [00:00<?, ?it/s]

  Found 12 unique time coordinate group(s):
    - 151 timesteps (1950-01-01T00:00:00.000000000 to 2100-01-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-02-01T00:00:00.000000000 to 2100-02-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-03-01T00:00:00.000000000 to 2100-03-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-04-01T00:00:00.000000000 to 2100-04-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-05-01T00:00:00.000000000 to 2100-05-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-06-01T00:00:00.000000000 to 2100-06-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-07-01T00:00:00.000000000 to 2100-07-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-08-01T00:00:00.000000000 to 2100-08-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-09-01T00:00:00.000000000 to 2100-09-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-10-01T00:00:00.000000000 to 2100-10-01T00:00:00.000000000): 2 file

...

  Found 12 unique time coordinate group(s):
    - 151 timesteps (1950-01-01T00:00:00.000000000 to 2100-01-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-02-01T00:00:00.000000000 to 2100-02-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-03-01T00:00:00.000000000 to 2100-03-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-04-01T00:00:00.000000000 to 2100-04-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-05-01T00:00:00.000000000 to 2100-05-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-06-01T00:00:00.000000000 to 2100-06-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-07-01T00:00:00.000000000 to 2100-07-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-08-01T00:00:00.000000000 to 2100-08-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-09-01T00:00:00.000000000 to 2100-09-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-10-01T00:00:00.000000000 to 2100-10-01T00:00:00.000000000): 2 file

  Loading res 5:   0%|          | 0/24 [00:00<?, ?it/s]

  Found 12 unique time coordinate group(s):
    - 151 timesteps (1950-01-01T00:00:00.000000000 to 2100-01-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-02-01T00:00:00.000000000 to 2100-02-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-03-01T00:00:00.000000000 to 2100-03-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-04-01T00:00:00.000000000 to 2100-04-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-05-01T00:00:00.000000000 to 2100-05-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-06-01T00:00:00.000000000 to 2100-06-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-07-01T00:00:00.000000000 to 2100-07-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-08-01T00:00:00.000000000 to 2100-08-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-09-01T00:00:00.000000000 to 2100-09-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-10-01T00:00:00.000000000 to 2100-10-01T00:00:00.000000000): 2 file

  Loading res 6:   0%|          | 0/24 [00:00<?, ?it/s]

  Found 12 unique time coordinate group(s):
    - 151 timesteps (1950-01-01T00:00:00.000000000 to 2100-01-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-02-01T00:00:00.000000000 to 2100-02-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-03-01T00:00:00.000000000 to 2100-03-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-04-01T00:00:00.000000000 to 2100-04-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-05-01T00:00:00.000000000 to 2100-05-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-06-01T00:00:00.000000000 to 2100-06-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-07-01T00:00:00.000000000 to 2100-07-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-08-01T00:00:00.000000000 to 2100-08-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-09-01T00:00:00.000000000 to 2100-09-01T00:00:00.000000000): 2 file(s)
    - 151 timesteps (1950-10-01T00:00:00.000000000 to 2100-10-01T00:00:00.000000000): 2 file

## Step 7: Extract Variable Metadata from Zarr Store

Extract metadata directly from Zarr store attributes (inherited from NetCDF).


In [58]:
def extract_variable_metadata_from_zarr(zarr_store_path):
    """
    Extract metadata from hierarchical Zarr store attributes.

    Args:
        zarr_store_path: Path to hierarchical Zarr store

    Returns:
        dict: Metadata by variable name (title, description, units, etc.)
    """
    print(f"Extracting metadata from Zarr attributes...")

    metadata = {}

    zarr_root = zarr.open_group(zarr_store_path, mode='r')
    available_groups = sorted([g for g in zarr_root.group_keys()], reverse=True)

    if not available_groups:
        print(f"  ⚠️ No groups found in Zarr store")
        return metadata

    first_group = available_groups[0]
    print(f"  Reading metadata from group: {first_group}")

    ds = xr.open_zarr(zarr_store_path, group=first_group, decode_timedelta=False)

    for var_name in ds.data_vars:
        var_attrs = ds[var_name].attrs

        metadata[var_name] = {
            "title": var_attrs.get("long_name", var_name),
            "description": var_attrs.get("description", ""),
            "x-ogc-unit": var_attrs.get("units", ""),
            "url": None
        }

        if "standard_name" in var_attrs:
            std_name = var_attrs["standard_name"]
            metadata[var_name]["x-ogc-definition"] = (
                f"http://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html#{std_name}"
            )

    ds.close()

    print(f"  ✅ Extracted metadata for {len(metadata)} variables from Zarr attributes")

    return metadata


In [59]:
if 'H3_RESOLUTION' not in globals() or H3_RESOLUTION is None:
    print("H3_RESOLUTION not defined. Run previous cells first.")
elif not FINAL_ZARR_PATH.exists():
    print(f"⚠️  Hierarchical Zarr store not found: {FINAL_ZARR_PATH}")
    print(f"    Run previous cells first!")
else:
    print(f"\n{'='*70}")
    print(f"EXTRACTING ZARR METADATA")
    print(f"{'='*70}\n")

    ZARR_METADATA = extract_variable_metadata_from_zarr(FINAL_ZARR_PATH)

    # Display in DataFrame
    if ZARR_METADATA:
        zarr_df = pd.DataFrame.from_dict(ZARR_METADATA, orient='index')
        zarr_df.index.name = 'variable'
        zarr_df = zarr_df.reset_index()

        print(f"\n{'='*120}")
        print(f"Zarr Variable Metadata ({len(zarr_df)} variables)")
        print(f"{'='*120}\n")

        pd.set_option('display.max_rows', None)
        pd.set_option('display.max_colwidth', 80)
        pd.set_option('display.width', None)

        display(zarr_df[['variable', 'title', 'description', 'x-ogc-unit']])
    else:
        print("⚠️ No metadata extracted from zarr")



EXTRACTING ZARR METADATA

Extracting metadata from zarr attributes...
  Reading metadata from group: res6
  ✅ Extracted metadata for 24 variables from zarr attributes

Zarr Variable Metadata (24 variables)



,variable,title,description,x-ogc-unit
0,ssp126_prcptot_p10,Total accumulated precipitation,Monthly total precipitation. 10th percentile of ensemble.,mm
1,ssp126_prcptot_p50,Total accumulated precipitation,Monthly total precipitation. 50th percentile of ensemble.,mm
2,ssp126_prcptot_p90,Total accumulated precipitation,Monthly total precipitation. 90th percentile of ensemble.,mm
3,ssp126_tx_max_p10,Maximum daily maximum temperature,Monthly maximum of daily maximum temperature. 10th percentile of ensemble.,K
4,ssp126_tx_max_p50,Maximum daily maximum temperature,Monthly maximum of daily maximum temperature. 50th percentile of ensemble.,K
5,ssp126_tx_max_p90,Maximum daily maximum temperature,Monthly maximum of daily maximum temperature. 90th percentile of ensemble.,K
6,ssp245_prcptot_p10,Total accumulated precipitation,Monthly total precipitation. 10th percentile of ensemble.,mm
7,ssp245_prcptot_p50,Total accumulated precipitation,Monthly total precipitation. 50th percentile of ensemble.,mm
8,ssp245_prcptot_p90,Total accumulated precipitation,Monthly total precipitation. 90th percentile of ensemble.,mm
9,ssp245_tx_max_p10,Maximum daily maximum temperature,Monthly maximum of daily maximum temperature. 10th percentile of ensemble.,K


## Step 8: [OPTIONAL] Augment Metadata from ClimateData.ca

This step is **optional**. It scrapes additional metadata from ClimateData.ca to augment
the metadata extracted from Zarr files. Skip this step if you don't need web-based metadata.

The scraped data includes:
- Human-friendly titles and descriptions for each variable
- URLs to ClimateData.ca variable pages


In [60]:
%%capture
!pip install selenium

In [61]:
def scrape_all_climatedata_variables():
    """
    Scrape ALL climate variable metadata from ClimateData.ca using `selenium`.

    This function is completely independent of Zarr files and scrapes all available
    variables from the ClimateData.ca website. Results are cached for reuse.

    Returns:
        dict: Complete metadata from ClimateData.ca for all available variables
    """
    cache_file = CACHE_DIR / "climatedata_all_variables.json"

    # Check cache first
    if cache_file.exists():
        print(f"Loading cached ClimateData.ca metadata from {cache_file}")
        with open(cache_file, mode="r", encoding="utf-8") as f:
            cached_data = json.load(f)
        print(f"  ✅ Loaded {len(cached_data)} variables from cache")
        return cached_data

    # No cache - scrape from website
    try:
        from selenium import webdriver
        from selenium.webdriver.chrome.options import Options
    except ImportError:
        print("⚠️  `selenium` not installed. Install with: pip install selenium")
        return {}

    print("Scraping ALL climate variables from ClimateData.ca...")
    print("  This is a one-time operation - results will be cached")

    options = Options()
    options.add_argument('--headless')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--disable-gpu')

    driver = webdriver.Chrome(options=options)

    try:
        print("  Loading https://climatedata.ca/variables/...")
        driver.get("https://climatedata.ca/variables/")

        print("  Waiting for JavaScript to render...")
        time.sleep(10)

        # Extract all variable information from rendered page
        script = """
        let vars = [];
        document.querySelectorAll('[id^="v-"]').forEach(el => {
            let id = el.id;
            let title = el.querySelector('h1, h2, h3, h4')?.textContent || '';
            let desc = el.querySelector('p')?.textContent || '';
            let exportName = '';

            // Look for "Name in exports"
            let textContent = el.textContent;
            let match = textContent.match(/Name in exports?:\\s*([a-z_-]+)/i);
            if (match) {
                exportName = match[1];
            }

            if (id) {
                vars.push({
                    id: id,
                    title: title.trim(),
                    description: desc.trim(),
                    export_name: exportName
                });
            }
        });
        return vars;
        """

        variables = driver.execute_script(script)
        print(f"  ✅ Scraped {len(variables)} variables from ClimateData.ca")

        # Convert to dict keyed by export_name
        metadata = {}
        for var in variables:
            export_name = var['export_name'] if var['export_name'] else var['id'].replace('v-', '').replace('-2', '')
            metadata[export_name] = {
                "title": var['title'],
                "description": var['description'],
                "url": f"https://climatedata.ca/variables/?#{var['id']}",
                "variable_id": export_name,
                "climatedata_id": var['id']
            }

        # Cache the results
        cache_file.parent.mkdir(parents=True, exist_ok=True)
        with open(cache_file, mode="w", encoding="utf-8") as f:
            json.dump(metadata, f, indent=2)
        print(f"  ✅ Cached to: {cache_file}")

        return metadata

    finally:
        driver.quit()


In [62]:
print(f"\n{'='*120}")
print(f"[OPTIONAL] SCRAPING ALL CLIMATEDATA.CA VARIABLES")
print(f"{'='*120}\n")
print("This step scrapes ALL available climate variables from ClimateData.ca.")
print("Results are cached and reused regardless of which variables you process.\n")

WEB_METADATA = scrape_all_climatedata_variables()

if WEB_METADATA:
    # Display metadata in DataFrame
    metadata_df = pd.DataFrame.from_dict(WEB_METADATA, orient='index')
    metadata_df.index.name = 'variable'
    metadata_df = metadata_df.reset_index()

    print(f"\n{'='*120}")
    print(f"ClimateData.ca Variable Metadata ({len(metadata_df)} variables)")
    print(f"{'='*120}\n")

    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_colwidth', None)
    pd.set_option('display.width', None)

    display(metadata_df[['variable', 'title', 'description', 'url']])
else:
    print("⚠️ Failed to scrape ClimateData.ca metadata")
    WEB_METADATA = None



[OPTIONAL] SCRAPING ALL CLIMATEDATA.CA VARIABLES

This step scrapes ALL available climate variables from ClimateData.ca.
Results are cached and reused regardless of which variables you process.

Loading cached ClimateData.ca metadata from /misc/scratch13/OGC11/canada-climate/cache/climatedata_all_variables.json
  ✅ Loaded 38 variables from cache

ClimateData.ca Variable Metadata (38 variables)



,variable,title,description,url
0,tx_max,Hottest Day,"The Hottest Day describes the warmest daytime temperature in the selected time period. In general, the hottest day of the year occurs during the summer months.",https://climatedata.ca/variables/?#v-tx_max-2
1,tg_mean,Mean Temperature,Mean temperature describes the average temperature for the 24-hour day.,https://climatedata.ca/variables/?#v-tg_mean-2
2,tn_mean,Minimum Temperature,"Minimum temperature describes the coldest temperature of the 24-hour day. Typically, but not always, the minimum temperature occurs at night and so this variable is commonly referred to as the nighttime low.",https://climatedata.ca/variables/?#v-tn_mean
3,tx_mean,Maximum Temperature,"Maximum temperature describes the warmest temperature of the 24-hour day. Typically, but not always, the maximum temperatures occur during the day and so this variable is commonly referred to as the daytime high.",https://climatedata.ca/variables/?#v-tx_mean-2
4,hxmax30,Days with Humidex above threshold,"Day with Maximum Humidex > x describes the number of days where the Maximum Humidex is greater than a threshold value, x. To view on the map select x from the available threshold values, or enter your preferred value for data download.",https://climatedata.ca/variables/?#v-hxmax30-2
5,days-below-temperature-threshold,Days below Tmin,"Days below Tmin describes the number of days where daily minimum temperature is below a threshold value, x. To view on the map select x from the available threshold values, or enter your preferred value for data download. For this index, threshold values are often selected to look at cold days where minimum temperatures are below, for example, -15°C, or -30°C. This index gives an indication of the number of very cold days in the selected time period.",https://climatedata.ca/variables/?#v-days-below-temperature-threshold
6,days-with-tmax-above-threshold,Days above Tmax,"Days above Tmax describes the number of days where daily maximum temperature is above a threshold value, x. To view on the map select x from the available threshold values, or enter your preferred value for data download. For this index, threshold values are often selected to look at hot days where maximum temperatures are above, for example, 25°C, 30°C or 35°C.",https://climatedata.ca/variables/?#v-days-with-tmax-above-threshold
7,days-above-tmax-and-tmin,Days above Tmax and Tmin,"Days above Tmax and Tmin describes the number of days where daily threshold values are exceeded for both maximum and minimum temperatures.This index can be used, for example, to identify extreme heat events where both daily maximum and minimum temperatures are chosen to represent hot conditions.",https://climatedata.ca/variables/?#v-days-above-tmax-and-tmin
8,tn_min,Coldest Day,"The Coldest Day describes the lowest nighttime temperature in the selected time period. In general, the coldest day of the year occurs during the winter months.",https://climatedata.ca/variables/?#v-tn_min-2
9,last_spring_frost,Last spring frost,The Last Spring Frost marks the approximate beginning of the growing season for frost-sensitive crops and plants. When the lowest temperature of the day remains above 0°C for one consecutive day (before July 15th) the date of the last spring frost is established.,https://climatedata.ca/variables/?#v-last_spring_frost-2


## Step 9: Generate `pydggsapi` Configuration

This step generates the final `pydggsapi` configuration using:
- Zarr metadata (required) - extracted in Step 7
- Web metadata (optional) - scraped in Step 8, if available
- Additional information such as spatial and temporal extent computed from the Zarr data
- Collection provider definitions aligned with the Zarr store structure

The configuration will work with Zarr metadata alone, but will be enhanced if web metadata is available.


In [66]:
def merge_metadata_sources(zarr_metadata, web_metadata=None):
    """
    Merge metadata from Zarr attributes and optionally from web scraping.

    Args:
        zarr_metadata: Metadata from Zarr attributes (required)
        web_metadata: Metadata from ClimateData.ca (optional)

    Returns:
        dict: Merged metadata with best available information
    """
    if not zarr_metadata:
        return {}

    merged = {}

    for var_name, zarr_meta in zarr_metadata.items():
        # Start with Zarr metadata
        merged[var_name] = zarr_meta.copy()

        # Optionally augment with web metadata if available
        if web_metadata:
            var_name_lower = var_name.lower()

            # Extract base variable name (remove ssp, percentile suffixes)
            # e.g., "ssp126_prcptot_p10" → "prcptot"
            parts = var_name_lower.split("_")
            base_candidates = []

            for i in range(len(parts)):
                for j in range(i+1, len(parts)+1):
                    candidate = "_".join(parts[i:j])
                    base_candidates.append(candidate)

            # Try to find a match in web metadata
            best_match = None
            best_match_len = 0

            for web_var_name in web_metadata:
                web_var_lower = web_var_name.lower()
                web_var_id = web_metadata[web_var_name].get("variable_id", "").lower()

                for candidate in base_candidates:
                    if candidate == web_var_lower or candidate == web_var_id:
                        if len(candidate) > best_match_len:
                            best_match = web_var_name
                            best_match_len = len(candidate)
                    elif candidate in web_var_lower or candidate in web_var_id:
                        if len(candidate) > best_match_len:
                            best_match = web_var_name
                            best_match_len = len(candidate)

            if best_match:
                web_meta = web_metadata[best_match]
                # Augment with web metadata (web takes precedence for description/URL)
                if web_meta.get("url"):
                    merged[var_name]["url"] = web_meta["url"]
                if web_meta.get("description"):
                    merged[var_name]["description"] = web_meta["description"]
                if web_meta.get("title") and not merged[var_name].get("title"):
                    merged[var_name]["title"] = web_meta["title"]

    return merged


In [67]:
def compute_spatial_temporal_extent(zarr_path):
    """
    Compute spatial and temporal extent from hierarchical Zarr store.
    Uses the highest resolution data for spatial extent.

    Args:
        zarr_path: Path to hierarchical Zarr store

    Returns:
        tuple: (spatial_bbox, temporal_range)
    """
    print(f"\nComputing spatial and temporal extent...")

    zarr_root = zarr.open_group(zarr_path, mode="r")
    group_names = sorted(
        [g for g in zarr_root.group_keys()],
        key=lambda x: int(x.replace("res", "")),
        reverse=True
    )
    highest_res_group = group_names[0]
    print(f"  Using highest resolution group: {highest_res_group}")

    ds = xr.open_zarr(zarr_path, group=highest_res_group, decode_timedelta=False)

    # Get H3 cells and compute bounding box
    h3_ids = ds["h3_id"].values
    print(f"  Sampling {min(1000, len(h3_ids))} H3 cells for spatial extent...")

    lats = []
    lons = []
    for h3_int in h3_ids[:1000]:
        h3_str = h3.int_to_str(h3_int)
        lat, lon = h3.cell_to_latlng(h3_str)
        lats.append(lat)
        lons.append(lon)

    spatial_bbox = [
        float(np.min(lons)),
        float(np.min(lats)),
        float(np.max(lons)),
        float(np.max(lats))
    ]

    print(f"  Spatial extent: [{spatial_bbox[0]:.2f}, {spatial_bbox[1]:.2f}, {spatial_bbox[2]:.2f}, {spatial_bbox[3]:.2f}]")

    # Get temporal extent
    time_values = pd.to_datetime(ds["time"].values)
    time_coords = sorted([t.isoformat() for t in time_values])

    temporal_range = {
        "interval": [[time_coords[0], time_coords[-1]]],
        "grid": {
            "cellsCount": len(time_coords),
            "coordinates": time_coords
        }
    }

    print(f"  Temporal extent: {time_coords[0]} to {time_coords[-1]} ({len(time_coords)} timesteps)")

    ds.close()

    return spatial_bbox, temporal_range


In [68]:
def generate_pydggsapi_config(zarr_path, h3_resolution, metadata, spatial_bbox, temporal_range, output_path):
    """
    Generate `pydggsapi` configuration from hierarchical Zarr store.

    Args:
        zarr_path: Path to hierarchical Zarr store
        h3_resolution: Maximum H3 resolution (base resolution)
        metadata: Variable metadata dictionary (merged from Zarr + optional web)
        spatial_bbox: Spatial bounding box [minLon, minLat, maxLon, maxLat]
        temporal_range: Temporal range with interval and grid
        output_path: Output path for configuration file

    Returns:
        dict: Generated configuration
    """
    print(f"\nGenerating pydggsapi configuration...")

    today = datetime.date.today().isoformat()

    zarr_root = zarr.open_group(zarr_path, mode="r")
    groups = sorted([g for g in zarr_root.group_keys()])

    print(f"  Found {len(groups)} resolution groups: {groups}")

    # Get all data variable names
    first_group = groups[0]
    ds = xr.open_zarr(zarr_path, group=first_group, decode_timedelta=False)
    data_cols = sorted(list(ds.data_vars))
    ds.close()

    print(f"  Variables: {len(data_cols)}")

    # Build zone_groups mapping for all resolutions
    zone_groups = {str(i): f"res{i}" for i in range(h3_resolution + 1)}

    config = {
        "dggrs": {
            "1": {
                "H3": {
                    "title": "DGGRS H3",
                    "description": (
                        "H3 hexagonal hierarchical geospatial indexing system developed by Uber "
                        "(https://github.com/uber/h3, https://h3geo.org/)."
                    ),
                    "crs": "wgs84",
                    "shapeType": "hexagon",
                    "uri": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                    "definition_link": "https://www.opengis.net/def/dggrs/OGC/1.0/H3",
                    "defaultDepth": 0,
                    "classname": "h3_dggrs_provider.H3Provider"
                }
            }
        },
        "collections": {
            "1": {
                "canada-climatedata": {
                    "title": "Canada Climate Indices DGGS Collection",
                    "description": (
                        "Climate indices mapped to H3 DGGS cells, aggregated from NetCDF sources "
                        "provided by ClimateData.ca."
                    ),
                    "attribution": f"ClimateData.ca [Accessed on {today}]",
                    "extent": {
                        "spatial": {
                            "bbox": [spatial_bbox],
                            "crs": "http://www.opengis.net/def/crs/OGC/1.3/CRS84"
                        },
                        "temporal": {
                            "interval": temporal_range["interval"],
                            "grid": temporal_range["grid"],
                            "trs": "http://www.opengis.net/def/uom/ISO-8601/0/Gregorian"
                        }
                    },
                    "links": [
                        {
                            "rel": "about",
                            "href": "https://climatedata.ca/",
                            "type": "text/html",
                            "title": "ClimateData.ca - Canadian Climate Data Portal",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "about",
                            "href": "https://donneesclimatiques.ca/",
                            "type": "text/html",
                            "title": "Portail canadien des données climatiques",
                            "hreflang": "fr-CA"
                        },
                        {
                            "rel": "license",
                            "href": "https://creativecommons.org/licenses/by/4.0/legalcode",
                            "type": "text/html",
                            "title": "Creative Commons Attribution International (CC BY)",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "terms-of-service",
                            "href": "https://climatedata.ca/about/legal/terms/",
                            "type": "text/html",
                            "title": "ClimateData.ca Terms of Service",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "describedby",
                            "href": "https://climatedata.ca/variables/",
                            "type": "text/html",
                            "title": "ClimateData.ca Variables Documentation",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "convertedfrom",
                            "href": "https://climatedata.ca/",
                            "type": "text/html",
                            "title": "Original ClimateData.ca Dataset Portal",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "via",
                            "href": "https://github.com/crim-ca/ogc-dggs/tree/main/canada-climate",
                            "type": "text/html",
                            "title": "Source code employed to generate DGGS data from reference features",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "cite-as",
                            "href": "https://hirondelle.crim.ca/dggs-api/collections/canada-climatedata",
                            "type": "application/json",
                            "title": "Canada Climate Indices as DGGS Collection",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "sponsored",
                            "href": "https://climatedata.ca/about/citing-climatedata-ca/",
                            "type": "text/html",
                            "title": "ClimateData.ca Sponsorship and Funding",
                            "hreflang": "en-CA"
                        },
                        {
                            "rel": "author",
                            "href": "https://crim.ca/fr/",
                            "type": "text/html",
                            "hreflang": "fr-CA",
                            "title": "Centre de recherche informatique de Montréal (CRIM)"
                        },
                        {
                            "rel": "author",
                            "href": "https://crim.ca/en/",
                            "type": "text/html",
                            "hreflang": "en-CA",
                            "title": "Centre de recherche informatique de Montréal (CRIM)"
                        }
                    ],
                    "collection_provider": {
                        "providerId": "canada-climatedata",
                        "dggrsId": "H3",
                        "dggrs_zoneid_repr": "int",
                        "min_refinement_level": 0,
                        "max_refinement_level": h3_resolution,
                        "datasource_id": "canada-climatedata"
                    }
                }
            }
        },
        "collection_providers": {
            "1": {
                "canada-climatedata": {
                    "classname": "zarr_collection_provider.ZarrCollectionProvider",
                    "datasources": {
                        "canada-climatedata": {
                            "filepath": str(Path(zarr_path).absolute()),
                            "zone_groups": zone_groups,
                            "id_col": "h3_id",
                            "datetime_col": "time",
                            "data_cols": data_cols,
                            "exclude_data_cols": []
                        }
                    }
                }
            }
        }
    }

    # Write configuration
    output_path = Path(output_path)
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(config, f, indent=4)

    print(f"\n✅ Configuration written to: {output_path}")
    print(f"   Collection ID: canada-climatedata")
    print(f"   Variables: {len(data_cols)}")
    print(f"   Zone groups: {len(zone_groups)} (res0 to res{h3_resolution})")
    print(f"   Spatial extent: {spatial_bbox}")
    print(f"   Temporal extent: {temporal_range['interval'][0]}")

    return config


In [70]:
if 'H3_RESOLUTION' not in globals() or H3_RESOLUTION is None:
    print("H3_RESOLUTION not defined. Run previous cells first.")
elif not FINAL_ZARR_PATH.exists():
    print(f"⚠️  Hierarchical Zarr store not found: {FINAL_ZARR_PATH}")
    print(f"    Run previous cells first!")
elif 'ZARR_METADATA' not in globals():
    print(f"⚠️  Zarr metadata not loaded. Run previous cells first.!")
else:
    print(f"\n{'='*70}")
    print(f"GENERATING PYDGGSAPI CONFIGURATION")
    print(f"{'='*70}\n")

    # Step 1: Merge metadata sources
    if 'WEB_METADATA' in globals() and WEB_METADATA:
        print("Merging Zarr metadata with web metadata...")
        merged_metadata = merge_metadata_sources(ZARR_METADATA, WEB_METADATA)
        print(f"  ✅ Merged metadata from both sources")
    else:
        print("Using Zarr metadata only (web metadata not available)...")
        merged_metadata = ZARR_METADATA
        print(f"  ✅ Using {len(merged_metadata)} variables from Zarr")

    # Step 2: Compute spatial and temporal extent
    spatial_bbox, temporal_range = compute_spatial_temporal_extent(FINAL_ZARR_PATH)

    # Step 3: Generate configuration
    pydggsapi_config = generate_pydggsapi_config(
        zarr_path=FINAL_ZARR_PATH,
        h3_resolution=H3_RESOLUTION,
        metadata=merged_metadata,
        spatial_bbox=spatial_bbox,
        temporal_range=temporal_range,
        output_path=FINAL_CONFIG_PATH
    )

    print(f"\n{'='*70}")
    print(f"✅ PYDGGSAPI CONFIGURATION COMPLETE")
    print(f"{'='*70}")


GENERATING PYDGGSAPI CONFIGURATION

Merging zarr metadata with web metadata...
  ✅ Merged metadata from both sources

Computing spatial and temporal extent...
  Using highest resolution group: res6
  Sampling 1000 H3 cells for spatial extent...
  Spatial extent: [-113.33, 75.40, -92.23, 80.07]
  Temporal extent: 1950-01-01T00:00:00 to 2100-12-01T00:00:00 (1812 timesteps)

Generating pydggsapi configuration...
  Found 7 resolution groups: ['res0', 'res1', 'res2', 'res3', 'res4', 'res5', 'res6']
  Variables: 24

✅ Configuration written to: /misc/scratch13/OGC11/canada-climate/outputs/pydggsapi_config.json
   Collection ID: canada-climatedata
   Variables: 24
   Zone groups: 7 (res0 to res6)
   Spatial extent: [-113.33477458070789, 75.39792954905052, -92.23180074425481, 80.06851683737929]
   Temporal extent: ['1950-01-01T00:00:00', '2100-12-01T00:00:00']

✅ PYDGGSAPI CONFIGURATION COMPLETE
